In [1]:
import pandas as pd
df=pd.read_csv("../data/solar data.csv")

In [2]:
df.head()

,Date,Time,DateTime,Country,DC_Power,AC_Power,Ambient_Temperature,Module_Temperature,Irradiation,Day,Month,Hour,Daily_Yield,Total_Yield
0,2024-01-01,0,2024-01-01 00:00:00,Norway,0.0,0.0,-10.15,-3.33,0.039,1,1,0,145.722,145.722
1,2024-01-01,1,2024-01-01 01:00:00,Norway,0.0,0.0,-10.06,-3.50,0.014,1,1,1,145.722,291.444
2,2024-01-01,2,2024-01-01 02:00:00,Norway,0.0,0.0,-9.95,-5.17,0.025,1,1,2,145.722,437.166
3,2024-01-01,3,2024-01-01 03:00:00,Norway,0.0,0.0,-9.27,-4.53,0.000,1,1,3,145.722,582.888
4,2024-01-01,4,2024-01-01 04:00:00,Norway,0.0,0.0,-9.62,-6.34,0.000,1,1,4,145.722,728.610


In [3]:
df.dtypes

Date                    object
Time                     int64
DateTime                object
Country                 object
DC_Power               float64
AC_Power               float64
Ambient_Temperature    float64
Module_Temperature     float64
Irradiation            float64
Day                      int64
Month                    int64
Hour                     int64
Daily_Yield            float64
Total_Yield            float64
dtype: object

In [4]:
df["Date"]=pd.to_datetime(df["Date"])
df["DateTime"]=pd.to_datetime(df["DateTime"])

In [5]:
df=df.sort_values(by="DateTime").reset_index(drop=True)

In [6]:
df.dtypes

Date                   datetime64[ns]
Time                            int64
DateTime               datetime64[ns]
Country                        object
DC_Power                      float64
AC_Power                      float64
Ambient_Temperature           float64
Module_Temperature            float64
Irradiation                   float64
Day                             int64
Month                           int64
Hour                            int64
Daily_Yield                   float64
Total_Yield                   float64
dtype: object

## Seperate X and y

In [7]:
X=df.drop(columns=["AC_Power","DC_Power","Date","Time","DateTime","Daily_Yield","Total_Yield"])
y=df["AC_Power"]

In [8]:
X.columns

Index(['Country', 'Ambient_Temperature', 'Module_Temperature', 'Irradiation',
       'Day', 'Month', 'Hour'],
      dtype='object')

## Train Test Split

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=False)

In [10]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(28108, 7)
(7028, 7)
(28108,)
(7028,)


## Missing values

In [11]:
print("Missing values in training data\n",X_train.isnull().sum().sum())
print("Missing values in testing data\n",X_test.isnull().sum().sum())

Missing values in training data
 0
Missing values in testing data
 0


## Feature Engineering

#### Here we created a new feature named "Temperature_Difference" as it is the difference between the other two features module temperature and ambient temperature. 

In [12]:
X_train["Temperature_Difference"]=(X_train["Module_Temperature"]-X_train["Ambient_Temperature"])
X_test["Temperature_Difference"]=(X_test["Module_Temperature"]-X_test["Ambient_Temperature"])

In [13]:
X_train.head()

,Country,Ambient_Temperature,Module_Temperature,Irradiation,Day,Month,Hour,Temperature_Difference
0,Norway,-10.15,-3.33,0.039,1,1,0,6.82
1,Australia,8.32,14.89,0.015,1,1,0,6.57
2,Brazil,20.95,23.76,0.000,1,1,0,2.81
3,India,17.93,22.86,0.000,1,1,0,4.93
4,Norway,-10.06,-3.50,0.014,1,1,1,6.56


In [14]:
y_train.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: AC_Power, dtype: float64

In [15]:
print(X_train.shape)
print(y_train.shape)

(28108, 8)
(28108,)


## Numerical and Categorical columns

In [16]:
numerical_col=X_train.select_dtypes(include="number").columns
numerical_col

Index(['Ambient_Temperature', 'Module_Temperature', 'Irradiation', 'Day',
       'Month', 'Hour', 'Temperature_Difference'],
      dtype='object')

In [17]:
categorical_col=X_train.select_dtypes(include=["object","category"]).columns
categorical_col

Index(['Country'], dtype='object')

## Categorical Encoding

In [18]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(handle_unknown="ignore",sparse_output=False,dtype=int)
encoder

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'int'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


#### Applying encoding to training data

In [19]:
encoded_train=encoder.fit_transform(X_train[categorical_col])
encoded_train

array([[0, 0, 0, 1],
       [1, 0, 0, 0],
       [0, 1, 0, 0],
       ...,
       [0, 1, 0, 0],
       [0, 0, 0, 1],
       [1, 0, 0, 0]], shape=(28108, 4))

#### Encoding testing data

In [20]:
encoded_test=encoder.transform(X_test[categorical_col])
encoded_test

array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1],
       ...,
       [0, 0, 0, 1],
       [0, 1, 0, 0],
       [1, 0, 0, 0]], shape=(7028, 4))

#### Convert encoded arrays to DataFrames

In [21]:
encoded_train_df=pd.DataFrame(encoded_train,columns=encoder.get_feature_names_out(categorical_col),index=X_train.index)
encoded_train_df

,Country_Australia,Country_Brazil,Country_India,Country_Norway
0,0,0,0,1
1,1,0,0,0
2,0,1,0,0
3,0,0,1,0
4,0,0,0,1
...,...,...,...,...
28103,1,0,0,0
28104,0,0,1,0
28105,0,1,0,0
28106,0,0,0,1


In [22]:
encoded_test_df=pd.DataFrame(encoded_test,columns=encoder.get_feature_names_out(categorical_col),index=X_test.index)
encoded_test_df

,Country_Australia,Country_Brazil,Country_India,Country_Norway
28108,0,1,0,0
28109,1,0,0,0
28110,0,0,0,1
28111,0,0,1,0
28112,0,0,1,0
...,...,...,...,...
35131,1,0,0,0
35132,0,0,1,0
35133,0,0,0,1
35134,0,1,0,0


In [23]:
X_train_num=X_train[numerical_col].copy()
X_test_num=X_test[numerical_col].copy()

In [24]:
X_train_final = pd.concat([X_train_num, encoded_train_df],axis=1)
X_test_final = pd.concat([X_test_num, encoded_test_df],axis=1)
print("Training shape:",X_train_final.shape)
print("Testing shape:",X_test_final.shape)
print("Training missing values:",X_train_final.isnull().sum().sum())
print("Testing missing values:",X_test_final.isnull().sum().sum())

Training shape: (28108, 11)
Testing shape: (7028, 11)
Training missing values: 0
Testing missing values: 0


## Feature Scaling

In [25]:
from sklearn.preprocessing import StandardScaler
numerical_features=X_train.select_dtypes(include="number").columns
scaler=StandardScaler()

In [26]:
sca_train=scaler.fit_transform(X_train[numerical_features])
sca_train

array([[-2.94303208, -2.76580648, -0.74713771, ..., -1.55690068,
        -1.66089585,  1.62129289],
       [-1.23708576, -1.08859926, -0.81353658, ..., -1.55690068,
        -1.66089585,  1.39833631],
       [-0.0705399 , -0.27208839, -0.85503588, ..., -1.55690068,
        -1.66089585, -1.95493075],
       ...,
       [ 0.05137939,  0.20750818, -0.38471053, ...,  1.68084454,
         0.94029618,  1.51427373],
       [-2.21151639, -2.17758726, -0.85503588, ...,  1.68084454,
         0.94029618,  0.25679858],
       [-0.72908875, -0.70657896, -0.30724518, ...,  1.68084454,
         0.94029618,  0.19437074]], shape=(28108, 7))

In [27]:
sca_test=scaler.transform(X_test[numerical_features])
sca_test

array([[ 1.04026348e-01,  3.44483815e-02, -8.13536583e-01, ...,
         1.68084454e+00,  1.08480685e+00, -6.70700814e-01],
       [-7.11539761e-01, -7.09340549e-01, -8.46736019e-01, ...,
         1.68084454e+00,  1.08480685e+00, -1.83105516e-03],
       [-2.22629448e+00, -2.36169343e+00, -8.55035878e-01, ...,
         1.68084454e+00,  1.08480685e+00, -1.38416189e+00],
       ...,
       [-2.89962143e+00, -2.92321726e+00, -8.02470104e-01, ...,
         2.40034348e+00,  1.66284953e+00, -3.22888539e-01],
       [-1.08408765e-01, -2.90682474e-02, -8.27369681e-01, ...,
         2.40034348e+00,  1.66284953e+00,  7.65139602e-01],
       [-1.25186385e+00, -1.26718225e+00, -8.55035878e-01, ...,
         2.40034348e+00,  1.66284953e+00, -1.89114588e-01]],
      shape=(7028, 7))

In [28]:
sca_train_df=pd.DataFrame(sca_train,columns=numerical_features,index=X_train.index)
sca_test_df=pd.DataFrame(sca_test,columns=numerical_features,index=X_test.index)

In [29]:
X_train_processed=pd.concat([sca_train_df, encoded_train_df], axis=1)
X_test_processed=pd.concat([sca_test_df, encoded_test_df], axis=1)
print("Training shape:",X_train_processed.shape)
print("Testing shape:",X_test_processed.shape)
print("Training missing values:",X_train_processed.isnull().sum().sum())
print("Testing missing values:",X_test_processed.isnull().sum().sum())
print("Same columns:",X_train_processed.columns.equals(X_test_processed.columns))

Training shape: (28108, 11)
Testing shape: (7028, 11)
Training missing values: 0
Testing missing values: 0
Same columns: True


## Baseline model

In [30]:
baseline_prediction=[y_train.mean()]*len(y_test)
baseline_prediction

[np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float64(16.805757079834923),
 np.float6

##### The baseline is a reference point.Our regression models should perform better than this simple average prediction.

In [31]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np
print("MAE: ",mean_absolute_error(y_test,baseline_prediction))
print("MSE: ",mean_squared_error(y_test,baseline_prediction))
print("RMSE: ",np.sqrt(mean_squared_error(y_test,baseline_prediction)))
print("R2 Score: ",r2_score(y_test,baseline_prediction))

MAE:  18.024975900115816
MSE:  424.4116041464824
RMSE:  20.601252489751257
R2 Score:  -0.0025609417661061062
